## 第五章
* 本章主要围绕模型构建、参数访问和初始化、自定义层和块、模型读写、GPU加速等内容展开

### 5.1 层与块

- 很多时候设计架构的时候考虑的不是单层，而是更粗粒度的**块**，在网络设计的过程中"比单个层大"但“比整个模型小”的组件更有价值，"块"可以描述单个层、由多个层组成的组件或整个模型本身，通过定义代码来生成任意复杂度的块，可以通过简洁的代码实现复杂的神经网络。

- 从编程的角度来看，块需要由"类(class)"表示，类的任何子类都必须定义一个将其输入转换为输出的前向传播函数，并且必须存储任何必需的参数。最后，为了计算梯度，块必须具有反向传播函数。

- 下面回顾一下之前写的多层感知机的代码：

In [2]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20,256),
        nn.ReLU(),
        nn.Linear(256,10))

X = torch.rand(2,20) # [0,1)均匀分布的采样
net(X)

tensor([[ 0.0361,  0.0713,  0.0761, -0.2873, -0.1185,  0.0976, -0.2254, -0.0114,
         -0.0387, -0.1092],
        [ 0.0401,  0.0397,  0.0881, -0.2479, -0.0109,  0.0199, -0.2196,  0.0806,
         -0.0362, -0.0474]], grad_fn=<AddmmBackward0>)

- 上述的例子中，通过实例化nn.Sequential构建了网络模型，层内的执行顺序是作为参数传递的。nn.Sequential定义了一种特殊的Module，即在Pytorch中表示一个块的类，其中两个全连接层也是Linear的实例，Linear类本身也是Module的子类。
- 另外，使用net(X)调用模型来获得模型的输出，这实际上是net.__call__(X)的简写，这个前向传播的逻辑即将列表中的每个块连接在一起，将每个块的输出作为下一个块的输入。
- 根据我们前面的叙述，块需要完成一系列的基本功能：
    - 将输入数据作为其前向传播函数的参数
    - 通过前向传播函数来生成输出
    - 计算其输出关于输入的梯度，可通过其反向传播函数进行访问，这个过程通常是自动发生的
    - 存储和访问前向传播所需要的参数
    - 根据需要初始化模型参数
- 下面定义了一个MLP类继承表示块的类，实现只需要提供构造函数和前向传播函数

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20,256)
        self.out = nn.Linear(256,10)

    def forward(self,x):
        return self.out(F.relu(self.hidden(x)))

- 上述的前向传播函数中，以x作为输入，计算带有激活函数的隐藏表示，并输出未规范化的输出值。MLP这个实现中两个层都是实例变量。
- 除非需要实现一个新的运算符，否则都不需要担心反向传播函数或参数初始化，系统将自动生成这些组成。
- 最开始在定义两层的MLP时使用了nn.Sequential,Sequential类的设计是为了把其他模块串起来，如果希望构建一个自定义的MySequential，我们只需要实现两个关键函数：
    - 一个是将各个块追加到列表中的函数
    - 一个是前向传播函数，用于将输入按追加块的顺序传递给块组成的链条
- 这样的MySequential定义如下：

In [ ]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            self._modules[str(idx)] = module
    
    def forward(self,X):
        for block in self._modules.values():
            X = block(X)
        return X
    
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

# 这个实现中主要通过*args接收了多个层的设计，每个单独的block和layer顺序执行调用



tensor([[-0.0838,  0.0342, -0.0700, -0.1948,  0.0556,  0.0437,  0.2703, -0.1104,
          0.0591,  0.0681],
        [-0.1473,  0.0139, -0.1908, -0.2127,  0.0594,  0.0258,  0.3560, -0.1699,
          0.0439,  0.1870]], grad_fn=<AddmmBackward0>)

- 并不是所有的架构都是简单的顺序架构，回顾在前几章的内容中，Pytorch支持自定义复杂的控制流计算，因此在块的设计中我们也希望执行任意的数学运算
- 下面介绍一个随机运算的特例，即线性计算中有一部分参数是**常数参数**，这部分参数并不参与更新，也不是上一层的运算结果，例如：

In [7]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight = torch.rand((20,20), requires_grad=False)
        self.linear = nn.Linear(20,20)
    
    def forward(self,X):
        X = self.linear(X)
        X = F.relu(torch.mm(X,self.rand_weight) + 1)
        X = self.linear(X)
        while X.abs().sum() > 1:
            X /= 2
        return X

# 上面的模型定义中有几个需要注意的部分，
# 一个是+1的偏置，让更多的通过mm计算得到的结果经过relu能保持激活值
# 一个是self.linear的复用，forward中前后使用了两次self.linear，两次计算使用的是同一组参数，即两个全连接层共享参数
# 最后有一个是复杂的控制流调用，通过设置一定的终止条件，重复进行计算，即使是比较复杂的控制流，反向传播的自动计算模块也能很好的实现梯度捕捉

net = FixedHiddenMLP()
net(X)

tensor([[-0.0400, -0.0659, -0.0364,  0.0127, -0.0180, -0.0237, -0.0016, -0.0071,
          0.0161,  0.0052,  0.0038, -0.0235, -0.0160,  0.0256,  0.0415, -0.0152,
          0.0625,  0.0194,  0.0097,  0.0546],
        [-0.0389, -0.0499, -0.0283,  0.0138, -0.0114, -0.0113, -0.0076, -0.0028,
          0.0155, -0.0055, -0.0091, -0.0220, -0.0176,  0.0245,  0.0427, -0.0226,
          0.0539, -0.0027,  0.0174,  0.0316]], grad_fn=<DivBackward0>)

- 回顾Python语言的一些特性，类似C语言之类的纯编译型语言的标准执行流程是编译->汇编->链接->执行，即将编写的代码翻译成汇编语言，再有汇编语言编译成机器指令加上链接生成类似.exe文件之类的执行文件；但Python不同，现代的Python最常使用Cpython作为实现，CPython将编译和解释放在了一起，第一次运行代码的时候将Python代码编译为字节码，之后每次的运行都会逐行解释并执行
- 上述Python的执行方式导致Python有一个独特的性质，即全局解释器锁(Global Interpreter Lock),GIL这一机制限制了同一时刻只有一个线程能执行Python字节码，即使在多线程的环境下，GIL也会确保只有一个线程在运行Python代码，因此当出现计算密集型任务时，需要通过多进程、扩展为Numpy等方式绕过GIL，否则这一机制就会成为性能瓶颈
- 编写执行块的操作效率问题是编写块时需要注意的，回顾之前的代码，很多的自定义块都是在反复进行字典查找、代码执行和其他一些控制类的Python代码，这些就属于上述的计算密集型任务，因此后续需要围绕这一机制进行特定的优化

### 5.2 参数管理

- 超参数和架构选择完成之后，我们就进入训练阶段，训练阶段会根据预先制定的损失函数更新参数值，在训练后，更新后的参数需要参与未来的预测，有时这些参数十分有用，就比如一些场景中我们需要复用他们，将模型和参数保存下来，以便可以在其他软件中执行或者进行检查。
- 本小节的内容就包括：
    - 访问参数，用于调试、诊断和可视化
    - 参数初始化
    - 在不同模型组件间共享参数

#### 5.2.1 参数访问

In [12]:
import torch
import torch.nn as nn

net = nn.Sequential(nn.Linear(4,8), nn.ReLU(), nn.Linear(8,1))
X = torch.rand(size=(2,4))

# 可以通过索引来访问模型的任意层，模型本身就是一个列表
print(net[2].state_dict())
# 输出的结果显示这个线性层当中包含两类参数，即权重和偏置

# 如果要对参数执行任何操作，首先就需要能够访问底层的数值
print(type(net[2].bias)) 
# 输出结果为<class 'torch.nn.parameter.Parameter'>
# 根据上述的输出结果可以分析一下，nn.parameter是一个命名空间，nn.parameter.Parameter是上述命名空间当中的一个类，继承于torch.tensor，这个类当中的实例会被自动注册为模型的参数，并参与梯度计算和优化
# nn.Module.parameters()是一个方法，用于返回模型中所有的Parameter对象，是一个return的迭代器
# 我们需要nn.Parameter这种类的原因是使得nn.Module能够自动管理模型参数，nn.Parameter会自动注册为模型参数

# 因此我们需要更进一步提取这个类当中真正封装的数值，即：
print(net[2].bias.data)
print(net[2].weight.grad == True) # nn.Parameter会默认分配梯度

# 如果需要处理一个复杂的块，逐个访问参数会变得很麻烦，因此可以使用一些递归的方式来提取每个子块的参数，需要学习一下下面的这种写法：
print(*[(name, param.shape) for name,param in net[0].named_parameters()])

# 根据上面的name输出结果，因此在访问子块的参数时还可以像下面一样写:
print(net.state_dict()['2.bias'].data)

OrderedDict([('weight', tensor([[ 0.1778,  0.0604, -0.2300, -0.2706,  0.3428,  0.2420,  0.3445,  0.1305]])), ('bias', tensor([-0.1285]))])
<class 'torch.nn.parameter.Parameter'>
tensor([-0.1285])
False
('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
tensor([-0.1285])


In [13]:
# 如果整个网络由嵌套块构成时：
def block1():
    return nn.Sequential(nn.Linear(4,8),nn.ReLU(),
                         nn.Linear(8,4),nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        net.add_module(f'block{i}',block1())
    return net
# 注意add_module和需要命名的写法

rgnet = nn.Sequential(block2(), nn.Linear(4,1))

print(rgnet)
# 这样会输出整个网络的结构

# 也可以根据嵌套的索引来读取参数数值
print(rgnet[0][1][0].weight.data)

Sequential(
  (0): Sequential(
    (block0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)
tensor([[-0.1002, -0.3481,  0.3851,  0.0432],
        [-0.1238,  0.1116, -0.1565, -0.3211],
        [-0.0143, -0.4588,  0.1640,  0.1610],
        [-0.3073,

#### 5.2.2 参数初始化

- 默认情况下，Pytorch会根据参数的维度来均匀地初始化权重和偏置矩阵

In [ ]:
# Pytorch中内置了初始化器，可以调用内置初始化器进行初始化

def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight,mean=0,std=1)
        nn.init.zeros_(m.bias)

        # 在前几章还有另外一种写法：
        m.weight.data.normal_(mean=0,std=1)
        m.bias.data.fill_(0)

        # 这里在d2l中还介绍了直接将所有的值赋为相同值的写法：
        nn.init.constant_(m.weight,mean=0,std=1)

net.apply(init_normal)

# print(net[0].weight.data[0],net[0].bias.data[0])

In [ ]:
# 基于上面的思路，我们也可以对整个网络中的不同块设置不同的初始化

def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)
def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight,42)

net[0].apply(init_xavier)
net[2].apply(init_42)
print(net[0].weight.data[0])
print(net[2].weight.data)

- 但内置的初始器不一定能满足我们所有的初始化需求
- 下面的代码给予这样好一个初始化，任意权重参数都遵循下面的规则：
    $$
        \mathcal{w} ~ 
        \begin{cases}
            \mathit{U}(5,10) & 可能性\frac{1}{4} \\
            0 & 可能性\frac{1}{2} \\
            \mathit{U}(-10,-5) & 可能性\frac{1}{4} \\
        \end{cases}
    $$

In [ ]:
# 下面的函数实现.md中的自定义初始化
def my_init(m):
    if type(m) == nn.Linear:
        print("Init", *[(name, param.shape) for name,param in m.named_parameters()][0])
        nn.init.uniform_(m.weight,-10,10)
        m.weight.data *= m.weight.data.abs() >= 5

net.apply(my_init)
# 这里有一个逻辑需要注意，我们前面的apply都是在不可分的网络块中执行的，但实际上也可以直接对整个网络施加，继承于nn.Module的网络类都会自动嵌套识别
print(net[0].weight.data[:2])

- 有时我们希望在多个层间共享参数，即一个稠密层的参数用来设置另外一个层的参数

In [ ]:
# 给共享层一个名称，以便可以引用
shared = nn.Linear(8,8)

net = nn.Sequential(
    nn.Linear(4,8),nn.ReLU(),
    shared,nn.ReLU(),
    shared,nn.ReLU(),
    nn.Linear(8,1)
)

print(net[2].weight.data[0] == net[4].weight.data[0])
# tensor([True, True, True, True, True, True, True, True])
net[2].weight.data[0,0] = 100
print(net[2].weight.data[0] == net[4].weight.data[0])
# tensor([True, True, True, True, True, True, True, True])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])


### 5.3 延后初始化

- 网络架构在很多时候可以直接确定，但具体的输入输出维度很难确定
- 初始化参数时，也没有足够的信息来确定模型应该包含多少参数
- 基于上面的情况，需要更深入地研究初始化机制

In [ ]:
# 传统初始化：这里用到了下一章的卷积神经网络结构
# 下面的例子说明一个现象：普通的结构都需要提前定义
nn.Linear(in_features=784, out_features=128)
nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3)

# 纯延后初始化网络：完全不用计算输入维度
# 连续的网络层仍需要输出维度进行配合，因此输出维度仍需要指定
class LazyNet(nn.Module):
    def __init__(self):
        super().__init__()
        # 卷积块：只写输出通道，输入通道自动推导
        self.conv = nn.Sequential(
            nn.LazyConv2d(out_channels=32, kernel_size=3, padding=1),  # 无in_channels
            nn.LazyBatchNorm2d(),  # 无num_features
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        # 全连接块：只写输出维度，输入维度自动推导
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(128),  # 无in_features
            nn.ReLU(),
            nn.LazyLinear(10)   # 无in_features
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

# 初始化模型
model = LazyNet()


# 第一次前向传播前：无参数！
print("初始化前参数量：", sum(p.numel() for p in model.parameters()))  # 输出：0

# 模拟输入：任意形状的图像（比如3通道，28x28）
x = torch.randn(2, 3, 28, 28)  # (batch, channel, H, W)
output = model(x)  # 第一次前向传播 → 触发延后初始化

# 初始化后：参数自动创建完成
print("初始化后参数量：", sum(p.numel() for p in model.parameters()))
print("输出形状：", output.shape)

model = LazyNet()
# 先触发初始化
dummy_input = torch.randn(1, 3, 28, 28)
model(dummy_input)

# Lazy层默认使用Kaiming初始化，但也可以自定义初始化方法
for m in model.modules():
    if isinstance(m, nn.LazyConv2d):
        nn.init.xavier_uniform_(m.weight)  # 替换初始化方式
        nn.init.zeros_(m.bias)

### 5.4 自定义层

- 除了在目前Pytorch框架中已封装好的网络层，完全可以按照任务的需求自定义一些目前还不存在的层

#### 5.4.1 不带参数的层

- 有一些计算步骤是不需要参数的自定义层

In [ ]:
import torch
import torch.nn.functional as F
import torch.nn as nn

class CenterLayer(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self,X):
        return X - X.mean()

# 上述自定义层可以验证是否正常工作

layer = CenterLayer()
a = torch.FloatTensor([1,2,3,4,5])
print(layer(a))

net = nn.Sequential(nn.Linear(8,128), CenterLayer())

Y = net(torch.rand(4,8))

print(Y.mean())

# 如果CenterLayer功能正常，计算出来的结果就会是0，实际实现过程中因为浮点数存储的问题，造成最终的计算结果不是严格为0
# 这里输出的结果为tensor(xxxx, grad_fn=<MeanBackward0>)

tensor([-2., -1.,  0.,  1.,  2.])
tensor(-8.3819e-09, grad_fn=<MeanBackward0>)


- 这里介绍一下后续的grad_fn，grad_fn是一个torch.autograd.Function的子类实例，用于记录该张量的计算历史和生成该张量的操作，之所以要记录这些，就是为了自动微分机制的正常运行，这也是该机制的核心部分，根据笔者的理解，这样一环套一环，自动微分就可以反向传播一层一层地计算每个张量的微分
- 除了这里的<MeanBackward0>，常见的类型还有AddBackward0、SubBackward0、MulBackward0、DivBackward0、MeanBackward0、ReshapeAliasBackward、CatBackward、ReluBackward0、AddmmBackward等等，从类型的名字就可以看出来相关的具体计算过程

#### 5.4.2 带参数的层

- 带参数的层才是更多情况下需要的自定义层，这些参数可以通过训练进行调整，我们可以使用内置函数来创建这些参数

In [4]:
# 下面的例子中通过自定义实现一个全连接层

class Mylinear(nn.Module):
    def __init__(self, in_units, units):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_units,units))
        self.bias = nn.Parameter(torch.randn(units,))
    def forward(self,X):
        linear = torch.matmul(X, self.weight.data) + self.bias.data
        # 注册为nn.Parameter之后如果需要提取数据就要.data,默认这个类当中的参数需要梯度
        return F.relu(linear)

linear = Mylinear(5,3)
print(linear.weight)

# 通过自定义层构建网络块、模型
net = nn.Sequential(Mylinear(64,8), Mylinear(8,1))
print(net(torch.rand(2,64)))

Parameter containing:
tensor([[ 0.5442,  0.4475,  1.5749],
        [ 0.4959,  0.8755,  0.9642],
        [ 0.3489, -0.2011, -0.0018],
        [-0.0598, -0.0823,  0.8135],
        [-0.8254,  0.3404,  0.6524]], requires_grad=True)
tensor([[ 7.2541],
        [11.1535]])


### 5.5 读写文件

- 截止目前，我们讨论了如何处理数据、如何构建、训练和测试学习模型，但很多时候我们会面临一种状况：服务器电源可能在一个长期任务中随时断掉，这时就需要我们定期保存中间结果，方便可以从某一个环节继续当时的训练，因此这一小节解决的就是如何加载和存储权重向量和模型

#### 5.5.1 记载和保存张量

In [5]:
# 下面介绍的保存和加载单个张量
import torch
from torch import nn
import torch.nn.functional as F

x = torch.arange(4)
torch.save(x, "x-file") # torch.save需要喂入一个变量名

x2 = torch.load("x-file")
print(x2)

tensor([0, 1, 2, 3])


/var/folders/dl/s82h6bc93fx7147qvyyp3dtw0000gn/T/ipykernel_81038/3046910857.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x2 = torch.load("x-file")


In [7]:
# 下面是保存张量列表和字典等数据的展示

y = torch.zeros(4)
torch.save([x,y],"x-files")
x2, y2 = torch.load("x-files")
print(x2,y2)

mydict ={
    "x":x,
    "y":y
}
torch.save(mydict,"mydict")
mydict2 = torch.load("mydict")
print(mydict2)

tensor([0, 1, 2, 3]) tensor([0., 0., 0., 0.])
{'x': tensor([0, 1, 2, 3]), 'y': tensor([0., 0., 0., 0.])}


/var/folders/dl/s82h6bc93fx7147qvyyp3dtw0000gn/T/ipykernel_81038/277700008.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  x2, y2 = torch.load("x-files")
/var/folders/dl

#### 5.5.2 记载和保存模型参数